# Visoria Enhanced Attention Model Training
This notebook demonstrates how the improved machine learning model is trained using engineered relational and geometric features.

## 1. Feature Engineering
First, we load the raw dataset and engineer high-signal features like `face_area` and `face_phone_dist`.

In [ ]:
import pandas as pd
import numpy as np

# Load original dataset
df = pd.read_csv('data/attention_detection_dataset_v1.csv')

# Engineer Features
df['face_area'] = df['face_w'] * df['face_h']
df['phone_area'] = df['phone_w'] * df['phone_h']

# Distance between face and phone (if phone is present)
dist = np.sqrt((df['face_x'] - df['phone_x'])**2 + (df['face_y'] - df['phone_y'])**2)
df['face_phone_dist'] = np.where(df['phone'] == 1, dist, 9999.0)

# Phone proximity flag (is phone held near face?)
df['phone_near_face'] = np.where((df['phone'] == 1) & (df['face_phone_dist'] < df['face_w'] * 2.0), 1.0, 0.0)

# Reorder label to the end
cols = list(df.columns)
cols.remove('label')
cols.append('label')
df = df[cols]

# Save enhanced dataset
df.to_csv('data/attention_detection_dataset_v2.csv', index=False)
print('Enhanced dataset shape:', df.shape)
df.head()

## 2. Model Training
Now we train a Random Forest Classifier using the enhanced feature set.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import joblib

df_v2 = pd.read_csv('data/attention_detection_dataset_v2.csv')

# Prepare features and labels
X = df_v2.drop(columns=['label'])
y = df_v2['label']

# One-hot encode categorical features (pose)
X = pd.get_dummies(X, columns=['pose'])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf.fit(X_train_s, y_train)

# Evaluate
y_pred = rf.predict(X_test_s)
print('Model Evaluation:')
print(classification_report(y_test, y_pred))

# Save artifacts
joblib.dump(rf, 'artifacts/attention_model.pkl')
joblib.dump(scaler, 'artifacts/attention_scaler.pkl')
joblib.dump(list(X.columns), 'artifacts/attention_columns.pkl')
print('Model artifacts saved successfully!')

## 3. Explainability (SHAP)
Let's see which features the model relies on the most. The new engineered features should rank highly!

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test_s)

# Summary plot
shap.summary_plot(shap_values, X_test, plot_type='bar')